<a href="https://colab.research.google.com/github/seanyi-creator/ba-305-project/blob/main/01_permits_prep.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Workstream 1 — Permit data, leakage audit, pipeline and baselines
**Owner:** Sean Yi · **Question:** When a building permit is filed in San Francisco, can the details on the application predict whether the project will fail to finish (expire, be cancelled or be withdrawn) rather than be completed?

This notebook does five things the rest of the team builds on:
1. Defines the target and keeps only permits whose outcome is known.
2. Chooses which filing years to use, with evidence.
3. Audits every column for leakage and drops what would not be known at filing.
4. Builds the shared preprocessing pipeline (imputation, encoding, scaling) and the train/test split.
5. Reports the baselines every model must beat.

**For teammates:** copy the cell marked `SHARED CELL` into your notebook and run `X_train, X_test, y_train, y_test, preprocessor = get_permit_data()`. Everyone then models the same rows, the same split and the same features.

In [1]:
# --- Sean: setup — libraries and display settings ---
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn import set_config
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score

pd.set_option('display.max_columns', 50)
set_config(display='diagram')   # shows sklearn pipelines as a diagram in Colab

## Shared cell (teammates copy this one)
Everything below is defined once here and used by every step of this notebook. Each choice is explained in the step that justifies it.

In [ ]:
# --- Sean: SHARED CELL — load, target, filing-year filter, leakage drops, features, split, preprocessor ---
# Copy this whole cell into your notebook, then run:
#   X_train, X_test, y_train, y_test, preprocessor = get_permit_data()
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

PERMITS_URL = "https://raw.githubusercontent.com/rfordatascience/tidytuesday/main/data/2022/2022-07-05/sf_permits.csv"

# Statuses where the outcome is final. 'complete' = 0; every other final status = 1 (did not finish).
FINAL_STATUS = {'complete': 0, 'expired': 1, 'cancelled': 1, 'withdrawn': 1,
                'disapproved': 1, 'revoked': 1, 'denied': 1}
YEAR_MIN, YEAR_MAX = 2000, 2013   # filing years kept (see Step 2)
RANDOM_STATE = 42                 # everyone uses this, so the split is identical

# Columns recorded AFTER filing (leak the outcome) or that are only labels for a case (see Step 3).
LEAKAGE_COLS = ['status', 'status_date', 'completed_date', 'issued_date', 'permit_expiration_date',
                'first_construction_document_date', 'revised_cost']
ID_COLS = ['permit_number', 'record_id', 'block', 'lot', 'street_number', 'street_number_suffix',
           'street_name', 'street_suffix', 'unit', 'unit_suffix', 'location', 'description',
           'permit_creation_date', 'filed_date', 'date', 'zipcode', 'tidf_compliance',
           'permit_type', 'existing_construction_type', 'proposed_construction_type']
# description: free text, left out of the baseline feature set (an optional extension).
# zipcode: overlaps with neighborhood; permit_type / construction_type codes: duplicates of their text labels.
# tidf_compliance: 100% missing.

NUMERIC = ['log_estimated_cost', 'cost_is_placeholder', 'plansets',
           'existing_units', 'proposed_units', 'unit_change',
           'number_of_existing_stories', 'number_of_proposed_stories', 'story_change',
           'filing_year', 'filing_month']
CATEGORICAL = ['permit_type_definition', 'existing_use', 'proposed_use', 'use_changes',
               'existing_construction_type_description', 'proposed_construction_type_description',
               'structural_notification', 'site_permit', 'fire_only_permit',
               'voluntary_soft_story_retrofit', 'neighborhoods_analysis_boundaries',
               'supervisor_district']


def load_permits(url=PERMITS_URL):
    """Raw file, as downloaded."""
    return pd.read_csv(url, low_memory=False)


def resolved_permits(raw):
    """Keep permits with a final outcome, filed in YEAR_MIN..YEAR_MAX, and add the target y."""
    df = raw[raw['status'].isin(FINAL_STATUS)].copy()
    df['filed'] = pd.to_datetime(df['filed_date'])
    df = df[df['filed'].dt.year.between(YEAR_MIN, YEAR_MAX)]
    df['y'] = df['status'].map(FINAL_STATUS)          # 1 = did not finish
    return df


def add_features(df):
    """Features that are all known on the day the permit is filed."""
    out = df.copy()
    cost = out['estimated_cost']
    out['cost_is_placeholder'] = (cost <= 1).astype(int)          # $0/$1 entered as a placeholder
    out['log_estimated_cost'] = np.log10(cost.where(cost > 1))     # skewed: $1 to $500M+, so use log10
    out['unit_change'] = out['proposed_units'] - out['existing_units']
    out['story_change'] = out['number_of_proposed_stories'] - out['number_of_existing_stories']
    both_known = out['existing_use'].notna() & out['proposed_use'].notna()
    out['use_changes'] = (out['existing_use'] != out['proposed_use']).map({True: 'yes', False: 'no'}).where(both_known)
    out['filing_year'] = out['filed'].dt.year
    out['filing_month'] = out['filed'].dt.month
    out['supervisor_district'] = out['supervisor_district'].astype('Int64').astype(str).replace('<NA>', np.nan)
    for c in ['structural_notification', 'site_permit', 'fire_only_permit', 'voluntary_soft_story_retrofit']:
        out[c] = out[c].fillna('N')                                # blank means "not flagged"
    return out


def make_preprocessor():
    """Fit inside a Pipeline, so imputation and scaling learn from training folds only."""
    numeric = Pipeline([
        ('impute', SimpleImputer(strategy='median', add_indicator=True)),  # median + a 'was missing' flag
        ('scale', StandardScaler()),
    ])
    categorical = Pipeline([
        ('impute', SimpleImputer(strategy='constant', fill_value='missing')),
        ('onehot', OneHotEncoder(handle_unknown='infrequent_if_exist', min_frequency=50)),  # rare levels pooled
    ])
    return ColumnTransformer([('num', numeric, NUMERIC), ('cat', categorical, CATEGORICAL)])


def get_permit_data(raw=None, test_size=0.2):
    """One call for teammates: same rows, features, split and preprocessor for everyone."""
    raw = load_permits() if raw is None else raw
    df = add_features(resolved_permits(raw))
    X, y = df[NUMERIC + CATEGORICAL], df['y']
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, stratify=y, random_state=RANDOM_STATE)
    return X_train, X_test, y_train, y_test, make_preprocessor()

## Step 1 — Which permits have a known outcome?
A permit that is still *issued* or *filed* has no outcome yet. Labelling it either way would be guessing, so only final statuses are kept.

In [ ]:
# --- Sean: Step 1 — status counts and the target ---
raw = load_permits()
print('All permits:', raw.shape)

status_counts = raw['status'].value_counts()
final = status_counts[status_counts.index.isin(FINAL_STATUS)]
open_ = status_counts[~status_counts.index.isin(FINAL_STATUS)]
print('\nFinal statuses (kept):'); print(final.to_string())
print(f"\nStill open (dropped): {open_.sum():,} permits across {len(open_)} statuses, e.g. "
      f"{', '.join(open_.index[:4])}")

**Target:** `y = 1` if the permit did **not** finish (expired, cancelled, withdrawn, disapproved, revoked or denied), `y = 0` if it was completed. The city acts on the risky permits, so recall and precision should be measured on them. Disapproved, revoked and denied are only 139 permits; they are counted as "did not finish" because the planned work never happened.

## Step 2 — Which filing years?
Two things change with the filing year: how many permits are still open in 2018 (they had less time to resolve), and the share that did not finish.

In [ ]:
# --- Sean: Step 2 — outcome rates by filing year, to choose the years kept ---
yrs = pd.to_datetime(raw['filed_date']).dt.year
by_year = pd.DataFrame({
    'still open (share of all filed)': (~raw['status'].isin(FINAL_STATUS)).groupby(yrs).mean(),
    'did not finish (share of resolved)': raw.loc[raw['status'].isin(FINAL_STATUS), 'status']
        .ne('complete').groupby(yrs).mean(),
}).loc[2000:2018]

ax = by_year.plot(marker='o', figsize=(9, 4.5))
ax.axvspan(YEAR_MIN - 0.5, YEAR_MAX + 0.5, color='grey', alpha=0.12, label='years kept')
ax.set_xlabel('filing year'); ax.set_ylabel('share of permits')
ax.set_xticks(range(2000, 2019, 2)); ax.grid(alpha=0.3); ax.legend()
ax.set_title('Recent permits are still open, so their outcomes are unknown')
plt.tight_layout(); plt.show()
by_year.round(3)

**Reading the chart:** by 2018, about half of the permits filed that year are still open, against 3% of those filed in 2000. Keeping 2016–2018 would mostly keep the fast projects, which makes completion look better than it is (survivorship). Keeping **2000–2013** gives every permit at least five years to resolve.

The "did not finish" share also falls steadily, from 46% in 2000 to 17% in 2013. Part of that may be censoring and part may be real change in how permits are handled. **`filing_year` is therefore a feature** (it is known at filing), and the drift is a limitation to discuss: a model trained on older years may over-predict failure for new permits.

In [ ]:
# --- Sean: Step 2 — apply the filter ---
permits = add_features(resolved_permits(raw))
print(f'Resolved permits filed {YEAR_MIN}–{YEAR_MAX}: {len(permits):,}')
print(f"Did not finish (y = 1): {permits['y'].mean():.1%}")

## Step 3 — Leakage audit
The test is chronological: **would the city know this value on the day the permit is filed?** For each suspect column, the table compares the "did not finish" rate when the column is filled in versus empty. A huge gap from a column recorded after filing means the column carries the outcome.

In [ ]:
# --- Sean: Step 3 — leakage audit table ---
base = resolved_permits(raw)
suspects = {
    'completed_date': 'recorded when the work is signed off',
    'issued_date': 'set when the permit is issued, after review',
    'permit_expiration_date': 'set at issuance',
    'first_construction_document_date': 'recorded once construction documents are issued',
    'revised_cost': 'updated during plan review',
    'status_date': 'date of the current status (the outcome itself)',
}
rows = []
for col, why in suspects.items():
    filled = base[col].notna()
    rows.append({'column': col, 'when it is recorded': why,
                 'did not finish | filled': round(base.loc[filled, 'y'].mean(), 3),
                 'did not finish | empty': round(base.loc[~filled, 'y'].mean(), 3) if (~filled).any() else None,
                 'share empty': round(1 - filled.mean(), 3)})
leak_table = pd.DataFrame(rows)
leak_table

**What the table shows:**
- `completed_date` is a perfect leak: it exists only for completed permits.
- `issued_date` and `permit_expiration_date`: permits that were never issued almost never finish. Issuance happens after filing, so at filing nobody knows whether it will be issued.
- `revised_cost` is the subtle one. The column looks like an ordinary cost, but whether it was filled in reflects what happened during review, after filing. Its *missingness* reveals the outcome.

All of these, plus `status` itself, are dropped (`LEAKAGE_COLS` in the shared cell). Identifiers (`permit_number`, `record_id`, `block`, `lot`, street address, `location`) are dropped too: they label a case rather than describe it, so a model could only memorise them.

In [ ]:
# --- Sean: Step 3 — confirm none of the dropped columns reach the model ---
features = NUMERIC + CATEGORICAL
assert not set(features) & set(LEAKAGE_COLS + ID_COLS), 'a dropped column is still in the features'
print(f'{len(features)} features kept ({len(NUMERIC)} numeric, {len(CATEGORICAL)} categorical).')
print('Dropped for leakage:', ', '.join(LEAKAGE_COLS))

## Step 4 — Features and their data types
All features are on the application when it is filed.

In [ ]:
# --- Sean: Step 4 — data types and missing values of the chosen features ---
feature_info = pd.DataFrame({
    'type': ['numeric'] * len(NUMERIC) + ['categorical (nominal)'] * len(CATEGORICAL),
    'distinct values': [permits[c].nunique() for c in features],
    'share missing': [round(permits[c].isna().mean(), 3) for c in features],
}, index=features)
feature_info.sort_values('share missing', ascending=False)

In [ ]:
# --- Sean: Step 4 — estimated cost is extremely skewed, so the model uses log10(cost) ---
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
c = permits.loc[permits['estimated_cost'] > 1, 'estimated_cost']
axes[0].hist(c, bins=60); axes[0].set_title('estimated cost ($), raw'); axes[0].set_xlabel('$'); axes[0].set_ylabel('permits')
axes[1].hist(np.log10(c), bins=60); axes[1].set_title('log10(estimated cost)'); axes[1].set_xlabel('log10($)')
plt.suptitle('A few huge projects squash the raw scale; on log10 the spread is readable')
plt.tight_layout(); plt.show()
print(f"Placeholder costs of $0–$1: {permits['cost_is_placeholder'].sum():,} permits (flagged, cost set to missing)")

## Step 5 — Train/test split
An 80/20 split, stratified so both parts keep the same share of failed permits, with `random_state=42` for everyone. The test set is not touched until each model is tuned; tuning uses 5-fold cross-validation on the training set.

In [ ]:
# --- Sean: Step 5 — the shared split ---
X_train, X_test, y_train, y_test, preprocessor = get_permit_data(raw)
print(f'Train: {len(X_train):,} permits, {y_train.mean():.1%} did not finish')
print(f'Test:  {len(X_test):,} permits, {y_test.mean():.1%} did not finish')

## Step 6 — Preprocessing pipeline
- **Numbers:** fill missing values with the training median, add a "was missing" indicator (missingness can carry meaning: new construction has no existing units), then standardize.
- **Categories:** fill missing with the label "missing", then one-hot encode. Levels with fewer than 50 permits are pooled into one "infrequent" column so rare uses don't become noise.

Because this sits inside a `Pipeline`, `fit` only ever sees training data (or training folds in cross-validation), so no test information leaks into the imputer or the scaler.

In [ ]:
# --- Sean: Step 6 — the preprocessor, shown as a diagram ---
preprocessor

In [ ]:
# --- Sean: Step 6 — pipeline diagram for the report and slides ---
fig, ax = plt.subplots(figsize=(13, 2.4)); ax.axis('off')
boxes = [
    ('sf_permits.csv\n86,103 permits', 0.03),
    ('Keep final outcomes\nfiled 2000–2013\ntarget: did not finish', 0.20),
    ('Drop leakage +\nidentifier columns', 0.37),
    ('Engineer features\nlog cost, unit and\nstory change, use change', 0.54),
    ('Train/test split 80/20\nstratified · seed 42', 0.71),
    ('Pipeline (fit on train only)\nimpute → scale / one-hot\n→ model → P(did not finish)', 0.88),
]
for text, x in boxes:
    ax.text(x, 0.5, text, ha='center', va='center', fontsize=10,
            bbox=dict(boxstyle='round,pad=0.6', fc='#e8f0f8', ec='#3b6ea5'))
for (_, x1), (_, x2) in zip(boxes, boxes[1:]):
    ax.annotate('', xy=(x2 - 0.075, 0.5), xytext=(x1 + 0.075, 0.5), arrowprops=dict(arrowstyle='->', lw=1.5))
ax.set_title('From raw permits to a predicted risk of not finishing', fontsize=12)
plt.tight_layout(); plt.show()

## Step 7 — Baselines
Two models that ignore every input (scikit-learn's `DummyClassifier`):
- **Most frequent:** always predicts "completed".
- **Stratified:** guesses at random in the training proportions (the instructor's "random answer" baseline).

Any real model must beat both on the metrics we care about: AUC (ranking risky permits above safe ones) and recall and precision on "did not finish".

In [ ]:
# --- Sean: Step 7 — baseline scores on the test set ---
rows = []
for name, strategy in [('Always "completed"', 'most_frequent'), ('Random guess (stratified)', 'stratified')]:
    dummy = Pipeline([('prep', make_preprocessor()),
                      ('model', DummyClassifier(strategy=strategy, random_state=RANDOM_STATE))])
    dummy.fit(X_train, y_train)
    pred = dummy.predict(X_test); prob = dummy.predict_proba(X_test)[:, 1]
    rows.append({'baseline': name,
                 'accuracy': round(accuracy_score(y_test, pred), 3),
                 'recall (did not finish)': round(recall_score(y_test, pred, zero_division=0), 3),
                 'precision (did not finish)': round(precision_score(y_test, pred, zero_division=0), 3),
                 'ROC AUC': round(roc_auc_score(y_test, prob), 3)})
baselines = pd.DataFrame(rows)
baselines

**Reading the baselines:** always predicting "completed" already scores about 64% accuracy while catching **zero** failed permits, which is why accuracy alone cannot judge our models. Both baselines have an AUC of about 0.5: they cannot rank risky permits above safe ones at all.

## Step 8 — Imputation vs. dropping rows
The directives ask us to go beyond dropping missing values. To check the choice, the same simple model (a logistic regression used only as a yardstick here; the tuned models are in workstreams 2 and 3) is scored by 5-fold cross-validated AUC on the training set two ways: with imputation, and with every row that has a missing value dropped.

In [ ]:
# --- Sean: Step 8 — does imputing beat dropping incomplete rows? ---
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
yardstick = lambda: Pipeline([('prep', make_preprocessor()),
                              ('model', LogisticRegression(max_iter=2000))])

auc_impute = cross_val_score(yardstick(), X_train, y_train, cv=cv, scoring='roc_auc')
complete_rows = X_train.notna().all(axis=1)
auc_drop = cross_val_score(yardstick(), X_train[complete_rows], y_train[complete_rows], cv=cv, scoring='roc_auc')

pd.DataFrame({
    'approach': ['Impute (median / "missing" + indicator)', 'Drop rows with any missing value'],
    'training rows used': [len(X_train), int(complete_rows.sum())],
    'did not finish (share)': [round(y_train.mean(), 3), round(y_train[complete_rows].mean(), 3)],
    'CV ROC AUC (mean)': [round(auc_impute.mean(), 3), round(auc_drop.mean(), 3)],
    'CV ROC AUC (sd)': [round(auc_impute.std(), 3), round(auc_drop.std(), 3)],
})

**Reading the comparison:** dropping incomplete rows throws away about 20% of the training permits (about 49,700 → 39,500) for no gain in AUC, and the share of failed permits barely moves. There is also a practical reason to impute: a model trained only on complete rows has no rule for a **new** permit with a blank field, and about one in five applications has one. So the pipeline imputes.

Note the yardstick's AUC of about 0.64: there is real but modest signal in the application details. Expect the tuned models to improve on it, but not to reach 0.9. Reporting that honestly is a full-marks answer (rubric item 4).

## Hand-off to workstreams 2 and 3
1. Copy the **SHARED CELL** into your notebook.
2. Run `X_train, X_test, y_train, y_test, preprocessor = get_permit_data()`.
3. Build `Pipeline([('prep', preprocessor), ('model', YourModel())])` and tune it with `GridSearchCV(..., cv=5, scoring='roc_auc')` on `X_train, y_train` only.
4. Score on `X_test` once, at the end, and compare with the baselines table above.